In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import sys
import torch

print("=== JARVIS ENVIRONMENT CHECK ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"GPU memory: {props.total_memory / 1024**3:.2f} GiB")

print("\nEnvironment check complete.")

In [ ]:
%pip install -q -U transformers accelerate sentencepiece

In [ ]:
import torch
import transformers
import accelerate
import sentencepiece

print("JARVIS dependency check")
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("STATUS: PASS")

In [ ]:
from huggingface_hub import HfApi

MODEL_ID = "google/gemma-4-E4B-it"

try:
    info = HfApi().model_info(MODEL_ID)
    print("Model repository:", info.id)
    print("Repository accessible: YES")
    print("JARVIS model access check: PASS")
except Exception as e:
    print("Repository check failed:")
    print(type(e).__name__, str(e)[:500])

In [ ]:
import importlib.util
import torch

print("bitsandbytes installed:",
      importlib.util.find_spec("bitsandbytes") is not None)
print("CUDA available:", torch.cuda.is_available())

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {free / 1024**3:.2f} GiB free "
        f"of {total / 1024**3:.2f} GiB"
)

In [ ]:
import importlib.util

print("bitsandbytes available:",
      importlib.util.find_spec("bitsandbytes") is not None)

print("Transformers available:",
      importlib.util.find_spec("transformers") is not None)

print("PyTorch available:",
      importlib.util.find_spec("torch") is not None)

In [ ]:
%pip install -q bitsandbytes

In [ ]:
import torch
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("JARVIS dependency check: PASS")

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

print("Gemma 4 model class: AVAILABLE")
print("4-bit quantization config: AVAILABLE")

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit configuration created: PASS")
print("GPU count:", torch.cuda.device_count())

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU is disabled. In Kaggle Settings, enable GPU before loading Gemma.'
assert torch.cuda.device_count() >= 1, 'No CUDA GPU is attached to this Kaggle session.'
print('GPU preflight: PASS')
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E4B-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading JARVIS brain...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.eval()

print("JARVIS model loaded successfully.")
print("Model device:", model.device)
print("JARVIS BRAIN STATUS: LOADED")

In [ ]:
import torch

JARVIS_SYSTEM = """
You are JARVIS, a personal AI assistant created for Redline Interactive Development.
Your identity is JARVIS. Speak in a composed, intelligent, articulate,
professional and helpful manner, like a capable personal assistant.
Do not introduce yourself using the underlying model's brand.
Do not claim to have abilities or have completed actions you have not performed.
If asked specifically what model powers you, answer truthfully.
"""

messages = [
    {"role": "system", "content": JARVIS_SYSTEM},
    {
        "role": "user",
        "content": "Hello, JARVIS. Introduce yourself and tell me what you can help me with."
    },
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_length = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
    )

reply = processor.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
).strip()

print("JARVIS:")
print(reply)


In [ ]:
messages = [
    {"role": "system", "content": JARVIS_SYSTEM},
    {
        "role": "user",
        "content": (
            "JARVIS, explain in two or three sentences what you can do "
            "right now in this notebook, and what you cannot do without "
            "additional tools or setup. Be accurate and do not invent abilities."
        ),
    },
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_length = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
    )

reply = processor.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
).strip()

print("JARVIS FUNCTIONAL CHECK 2:")
print(reply)


In [ ]:
# Start the existing JARVIS API bridge using the Gemma model already loaded above.
import urllib.request

launcher_url = 'https://raw.githubusercontent.com/REDLINE-INTERACTIVE-DEV/JARVIS/main/notebook_bridge/kaggle_start_cell.py'
with urllib.request.urlopen(launcher_url, timeout=30) as response:
    launcher_source = response.read().decode('utf-8')
exec(compile(launcher_source, 'kaggle_start_cell.py', 'exec'), globals())
